# Model Generation for Allen Visual dataset

This notebook generates 12 models as used in the analysis and in the report. It is meant to be run on colab for GPU usage.
1. Load Data
2. Train Models:
  - Single: 1 untrained + 5 Trained (mouse 4)
  - Multi: 1 untrained + 5 Trained (all mice)
3. Saves the models in Drive

Notes: The untraind models had _UT at the end while the trained did not have anything yet. The function lens.model.model_loader() works with this because that's how the models were trained during the semester. Future models should be trained using _TR at the end to simplify the task and thus the function should be modified to take this into account. Another viable option could be to distinguish UT from trained using the number of steps: number of steps = 1 -> UT, else TR.

In [1]:
!ls

'ls' is not recognized as an internal or external command,
operable program or batch file.


In [1]:
!pip install --pre 'cebra[datasets,demos]'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 79.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 72.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 56.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 205.0/205.0 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/53.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
import os
from cebra import CEBRA
import sys
sys.path.append('/content/drive/MyDrive/mathis')
from GithubFolder.src.cebra_lens import cebra_lens as lens

os.environ["DATA_PATH"] = "/content/drive/MyDrive/mathis"


# Load Data

Access single session data using train_datas[i].neural and .index for the DINO features.

This code uses Celia's function get_single_session_datasets().

In [4]:
cd drive/MyDrive/mathis/

/content/drive/MyDrive/mathis


In [5]:
!ls

CEBRA_models  data  GithubFolder


In [6]:
train_datas, valid_datas, discrete_labels_train, discrete_labels_val = (
    lens.utils_allen.get_single_session_datasets()
)

parameters = {
    "lr": 3e-4,
    "model_architechture": "offset10-model",
    "num_units": 32,
    "output_dim": 128,
    "temp": 1,
    "time_offsets": 10,
    "batch_size": 2042,
}


mice = ["mouse1", "mouse2", "mouse3", "mouse4"]

/usr/local/lib/python3.11/dist-packages/cebra/datasets/allen/single_session_ca.py:246: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  frame_feature = torch.load(frame_feature

## Single session

In [7]:
max_iterations = 10000

In [14]:
# RUN MOUSE4 5 TIMES:
for i in range(5):

    cebra_model_single_session = CEBRA(
        model_architecture=parameters["model_architechture"],
        batch_size=parameters["batch_size"],
        learning_rate=parameters["lr"],
        temperature=parameters["temp"],
        num_hidden_units=parameters["num_units"],
        output_dimension=parameters["output_dim"],
        max_iterations=max_iterations,
        distance="cosine",
        conditional="time_delta",
        device="cuda",
        verbose=True,
        time_offsets=parameters["time_offsets"],
    )

    cebra_model_single_session.fit(train_datas[3].neural, train_datas[3].index)
    embeddings_single_session = cebra_model_single_session.transform(
        train_datas[3].neural
    )

    ################### SAVING ###################

    # torch version
    model_path = os.path.join(
        os.environ["DATA_PATH"],
        f"CEBRA_models/FinalModels/VISION/allen_single_session_{mice[3]}_{int(max_iterations/1000)}k_{i}_torch.pt",
    )
    cebra_model_single_session.save(model_path, backend="torch")
    print("Torch model saved under: ", model_path)
    # sklearn version
    model_path = os.path.join(
        os.environ["DATA_PATH"],
        f"CEBRA_models/FinalModels/VISION/allen_single_session_{mice[3]}_{int(max_iterations/1000)}k_{i}.pt",
    )
    cebra_model_single_session.save(model_path)
    print("Sklearn Model saved under: ", model_path)

pos: -0.9818 neg:  7.6587 total:  6.6769 temperature:  1.0000: 100%|██████████| 10000/10000 [01:33<00:00, 106.58it/s]


Torch model saved under:  /content/drive/MyDrive/mathis/CEBRA_models/FinalModels/VISION/allen_single_session_mouse4_10k_0_torch.pt
Sklearn Model saved under:  /content/drive/MyDrive/mathis/CEBRA_models/FinalModels/VISION/allen_single_session_mouse4_10k_0.pt


pos: -0.9812 neg:  7.6566 total:  6.6754 temperature:  1.0000: 100%|██████████| 10000/10000 [01:33<00:00, 106.68it/s]


Torch model saved under:  /content/drive/MyDrive/mathis/CEBRA_models/FinalModels/VISION/allen_single_session_mouse4_10k_1_torch.pt
Sklearn Model saved under:  /content/drive/MyDrive/mathis/CEBRA_models/FinalModels/VISION/allen_single_session_mouse4_10k_1.pt


pos: -0.9780 neg:  7.6564 total:  6.6785 temperature:  1.0000: 100%|██████████| 10000/10000 [01:34<00:00, 106.22it/s]


Torch model saved under:  /content/drive/MyDrive/mathis/CEBRA_models/FinalModels/VISION/allen_single_session_mouse4_10k_2_torch.pt
Sklearn Model saved under:  /content/drive/MyDrive/mathis/CEBRA_models/FinalModels/VISION/allen_single_session_mouse4_10k_2.pt


pos: -0.9768 neg:  7.6559 total:  6.6791 temperature:  1.0000: 100%|██████████| 10000/10000 [01:33<00:00, 106.51it/s]


Torch model saved under:  /content/drive/MyDrive/mathis/CEBRA_models/FinalModels/VISION/allen_single_session_mouse4_10k_3_torch.pt
Sklearn Model saved under:  /content/drive/MyDrive/mathis/CEBRA_models/FinalModels/VISION/allen_single_session_mouse4_10k_3.pt


pos: -0.9784 neg:  7.6559 total:  6.6776 temperature:  1.0000: 100%|██████████| 10000/10000 [01:33<00:00, 106.63it/s]


Torch model saved under:  /content/drive/MyDrive/mathis/CEBRA_models/FinalModels/VISION/allen_single_session_mouse4_10k_4_torch.pt
Sklearn Model saved under:  /content/drive/MyDrive/mathis/CEBRA_models/FinalModels/VISION/allen_single_session_mouse4_10k_4.pt


In [15]:
# UNTRAINED
max_iterations = 1

cebra_model_single_session = CEBRA(
    model_architecture=parameters["model_architechture"],
    batch_size=parameters["batch_size"],
    learning_rate=parameters["lr"],
    temperature=parameters["temp"],
    num_hidden_units=parameters["num_units"],
    output_dimension=parameters["output_dim"],
    max_iterations=max_iterations,
    distance="cosine",
    conditional="time_delta",
    device="cuda_if_available",
    verbose=True,
    time_offsets=parameters["time_offsets"],
)

cebra_model_single_session.fit(train_datas[3].neural, train_datas[3].index)
embeddings_single_session = cebra_model_single_session.transform(train_datas[3].neural)

################### SAVING ###################

# torch version
model_path = os.path.join(
    os.environ["DATA_PATH"],
    f"CEBRA_models/FinalModels/VISION/allen_single_session_{mice[3]}_{int(max_iterations/1000)}k_UT_torch.pt",
)
cebra_model_single_session.save(model_path, backend="torch")
print("Torch model saved under: ", model_path)
# sklearn version
model_path = os.path.join(
    os.environ["DATA_PATH"],
    f"CEBRA_models/FinalModels/VISION/allen_single_session_{mice[3]}_{int(max_iterations/1000)}k_UT.pt",
)
cebra_model_single_session.save(model_path)
print("Sklearn Model saved under: ", model_path)

pos: -0.8946 neg:  8.4866 total:  7.5921 temperature:  1.0000: 100%|██████████| 1/1 [00:00<00:00, 80.02it/s]

Torch model saved under:  /content/drive/MyDrive/mathis/CEBRA_models/FinalModels/VISION/allen_single_session_mouse4_0k_UT_torch.pt
Sklearn Model saved under:  /content/drive/MyDrive/mathis/CEBRA_models/FinalModels/VISION/allen_single_session_mouse4_0k_UT.pt


## Multi Session

In [11]:
untrained = False

multi_session_neural_train = []
multi_session_index_train = []
multi_session_neural_valid = []
multi_session_index_valid = []

for i in range(4):
    multi_session_neural_train.append(train_datas[i].neural)
    multi_session_index_train.append(train_datas[i].index)

    multi_session_neural_valid.append(valid_datas[i].neural)
    multi_session_index_valid.append(valid_datas[i].index)


max_iterations = 10000

In [12]:
# TRAIN 5 MULTI-SESSIONS

for i in range(5):
    # Multisession training
    cebra_model_multi_session = CEBRA(
        model_architecture=parameters["model_architechture"],
        batch_size=parameters["batch_size"],
        learning_rate=parameters["lr"],
        temperature=parameters["temp"],
        num_hidden_units=parameters["num_units"],
        output_dimension=parameters["output_dim"],
        max_iterations=max_iterations,
        distance="cosine",
        conditional="time_delta",
        device="cuda_if_available",
        verbose=True,
        time_offsets=parameters["time_offsets"],
    )

    # Provide a list of data, i.e. datas = [data_a, data_b, ...]
    cebra_model_multi_session.fit(multi_session_neural_train, multi_session_index_train)

    ################### SAVING ###################

    # torch version
    model_path = os.path.join(
        os.environ["DATA_PATH"],
        f"CEBRA_models/FinalModels/VISION/allen_multi_session_{int(max_iterations/1000)}k_{i}_torch.pt",
    )
    cebra_model_multi_session.save(model_path, backend="torch")
    print("Torch Model saved under: ", model_path)
    # sklearn version
    model_path = os.path.join(
        os.environ["DATA_PATH"],
        f"CEBRA_models/FinalModels/VISION/allen_multi_session_{int(max_iterations/1000)}k_{i}.pt",
    )
    cebra_model_multi_session.save(model_path)
    print("Sklearn Model saved under: ", model_path)

pos: -0.9807 neg:  9.0710 total:  8.0902 temperature:  1.0000: 100%|██████████| 10000/10000 [11:58<00:00, 13.92it/s]


Torch Model saved under:  /content/drive/MyDrive/mathis/CEBRA_models/FinalModels/VISION/allen_multi_session_10k_0_torch.pt
Sklearn Model saved under:  /content/drive/MyDrive/mathis/CEBRA_models/FinalModels/VISION/allen_multi_session_10k_0.pt


pos: -0.9852 neg:  9.0727 total:  8.0876 temperature:  1.0000: 100%|██████████| 10000/10000 [12:00<00:00, 13.89it/s]


Torch Model saved under:  /content/drive/MyDrive/mathis/CEBRA_models/FinalModels/VISION/allen_multi_session_10k_1_torch.pt
Sklearn Model saved under:  /content/drive/MyDrive/mathis/CEBRA_models/FinalModels/VISION/allen_multi_session_10k_1.pt


pos: -0.9839 neg:  9.0666 total:  8.0828 temperature:  1.0000: 100%|██████████| 10000/10000 [11:59<00:00, 13.89it/s]


Torch Model saved under:  /content/drive/MyDrive/mathis/CEBRA_models/FinalModels/VISION/allen_multi_session_10k_2_torch.pt
Sklearn Model saved under:  /content/drive/MyDrive/mathis/CEBRA_models/FinalModels/VISION/allen_multi_session_10k_2.pt


pos: -0.9811 neg:  9.0632 total:  8.0822 temperature:  1.0000: 100%|██████████| 10000/10000 [12:01<00:00, 13.86it/s]


Torch Model saved under:  /content/drive/MyDrive/mathis/CEBRA_models/FinalModels/VISION/allen_multi_session_10k_3_torch.pt
Sklearn Model saved under:  /content/drive/MyDrive/mathis/CEBRA_models/FinalModels/VISION/allen_multi_session_10k_3.pt


pos: -0.9841 neg:  9.0722 total:  8.0880 temperature:  1.0000: 100%|██████████| 10000/10000 [11:59<00:00, 13.90it/s]


Torch Model saved under:  /content/drive/MyDrive/mathis/CEBRA_models/FinalModels/VISION/allen_multi_session_10k_4_torch.pt
Sklearn Model saved under:  /content/drive/MyDrive/mathis/CEBRA_models/FinalModels/VISION/allen_multi_session_10k_4.pt


In [9]:
# UNTRAINED

max_iterations = 1

# Multisession training
cebra_model_multi_session = CEBRA(
    model_architecture=parameters["model_architechture"],
    batch_size=parameters["batch_size"],
    learning_rate=parameters["lr"],
    temperature=parameters["temp"],
    num_hidden_units=parameters["num_units"],
    output_dimension=parameters["output_dim"],
    max_iterations=max_iterations,
    distance="cosine",
    conditional="time_delta",
    device="cuda_if_available",
    verbose=True,
    time_offsets=parameters["time_offsets"],
)

# Provide a list of data, i.e. datas = [data_a, data_b, ...]
cebra_model_multi_session.fit(multi_session_neural_train, multi_session_index_train)

################### SAVING ###################

# torch version
model_path = os.path.join(
    os.environ["DATA_PATH"],
    f"CEBRA_models/FinalModels/VISION/allen_multi_session_{int(max_iterations/1000)}k_UT_torch.pt",
)
cebra_model_multi_session.save(model_path, backend="torch")
print("Torch Model saved under: ", model_path)
# sklearn version
model_path = os.path.join(
    os.environ["DATA_PATH"],
    f"CEBRA_models/FinalModels/VISION/allen_multi_session_{int(max_iterations/1000)}k_UT.pt",
)
cebra_model_multi_session.save(model_path)
print("Sklearn Model saved under: ", model_path)

pos: -0.2334 neg:  9.3212 total:  9.0878 temperature:  1.0000: 100%|██████████| 1/1 [00:01<00:00,  1.41s/it]

Torch Model saved under:  /content/drive/MyDrive/mathis/CEBRA_models/FinalModels/VISION/allen_multi_session_0k_UT_torch.pt
Sklearn Model saved under:  /content/drive/MyDrive/mathis/CEBRA_models/FinalModels/VISION/allen_multi_session_0k_UT.pt
